# CIFAR-10 Image Classification — Improved Baseline CNN

This run **fixes the low test accuracy (56%)** of notebook `01` and the shape bug from the
first version of this notebook.

### What was wrong
- Notebook `01`: heavy augmentation (`RandomRotation`/`Zoom`/`Translation`) **destroys 32x32
  images**, and augmentation was applied once (static snapshot) → plateau at ~56%.
- First version of `01b`: `ZeroPadding2D(4)` + `RandomCrop(32)` placed *inside* the model.
  At inference (`training=False`), Keras `RandomCrop` doesn't crop, so a `40x40` tensor reached
  the `Dense` layer (shape mismatch: expected 2048, got 3200).

### The fix (v2)
- **tf.data pipeline** applies augmentation only to the *training* set: pad 4 → random crop 32 → random flip.
- Validation/test slices go through the **clean model** unmodified — no shape surprises.
- EarlyStopping patience 8 (was 5), 40 epochs, LR reduction on plateau.

Expected result: **~85–90% test accuracy**.

In [ ]:
# Install extra packages (TensorFlow is pre-installed in Colab)
%pip install -q numpy matplotlib scikit-learn

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# ---------- Configuration ----------
BATCH_SIZE   = 64
EPOCHS       = 40
LEARNING_RATE = 0.001
PATIENCE     = 8
VAL_SPLIT    = 0.15
PAD          = 4
NUM_CLASSES  = 10

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

MODELS_DIR = "/content/models"
RESULTS_DIR = "/content/results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1) Data Pipeline

Load CIFAR-10, normalize to [0,1], one-hot labels, split 85/15 (test untouched).
Then build three `tf.data` datasets: the **training** one applies pad→crop→flip
augmentation; validation and test are left clean.

In [ ]:
def load_cifar10():
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    return x_train, y_train, x_test, y_test

def normalize(x):
    return x.astype("float32") / 255.0

def one_hot(y, num_classes=NUM_CLASSES):
    return keras.utils.to_categorical(y, num_classes)

def split_train_val(x, y, val_ratio=VAL_SPLIT, seed=SEED):
    indices = np.arange(len(x))
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    n_val = int(len(indices) * val_ratio)
    val_idx, train_idx = indices[:n_val], indices[n_val:]
    return x[train_idx], y[train_idx], x[val_idx], y[val_idx]

x_train, y_train, x_test, y_test = load_cifar10()
x_train, y_train, x_val, y_val = split_train_val(x_train, y_train)

x_train = normalize(x_train)
x_val   = normalize(x_val)
x_test  = normalize(x_test)
y_train = one_hot(y_train)
y_val   = one_hot(y_val)
y_test  = one_hot(y_test)

print("train:", x_train.shape, "| val:", x_val.shape, "| test:", x_test.shape)

In [ ]:
def make_train_ds(x, y, batch_size=BATCH_SIZE, pad=PAD, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    def augment(im, lbl):
        im = tf.pad(im, [[pad, pad], [pad, pad], [0, 0]])   # 32 -> 40
        im = tf.image.random_crop(im, (32, 32, 3), seed=seed)  # 40 -> 32
        im = tf.image.random_flip_left_right(im)
        return im, lbl

    ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(10000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def make_eval_ds(x, y, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_train_ds(x_train, y_train)
val_ds   = make_eval_ds(x_val, y_val)
test_ds  = make_eval_ds(x_test, y_test)

for batch in train_ds.take(1):
    print("train batch:", batch[0].shape, batch[1].shape)
for batch in val_ds.take(1):
    print("val batch:  ", batch[0].shape, batch[1].shape)

## 2) Visualize Sample Images

In [ ]:
def show_sample_grid(x, y, class_names=CLASS_NAMES, num_samples=15):
    cols = 5
    rows = int(np.ceil(num_samples / cols)) or 1
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2.4))
    axes = np.atleast_1d(axes).ravel()
    y_labels = np.argmax(y, axis=-1) if y.ndim == 2 else y
    for i in range(num_samples):
        axes[i].imshow(x[i])
        axes[i].set_title(class_names[y_labels[i]], fontsize=8)
        axes[i].axis("off")
    for ax in axes[num_samples:]:
        ax.axis("off")
    plt.suptitle("Sample Training Images", y=1.02)
    plt.tight_layout()
    plt.show()

show_sample_grid(x_train, y_train)

## 3) Build the CNN Model (clean — no augmented layers inside)

Augmentation lives in the training `tf.data` pipeline, so the model always sees
`32x32` inputs — training and inference behave identically.

In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cifar10_cnn_improved")
    return model

model = build_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 4) Train the Model

EarlyStopping patience 8, LR halves on plateau, checkpoint keeps best `val_accuracy`.
The model trains on the augmented `train_ds` and validates on the clean `val_ds`.

In [ ]:
callbacks_list = [
    callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, "img_class_best.keras"),
                              monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1,
)

In [ ]:
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history["loss"], label="train")
    ax1.plot(history.history["val_loss"], label="val")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True)
    ax2.plot(history.history["accuracy"], label="train")
    ax2.plot(history.history["val_accuracy"], label="val")
    ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "training_history.png"), dpi=120, bbox_inches="tight")
    plt.show()

plot_training_history(history)

## 5) Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_true = np.argmax(y_test, axis=-1)
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=-1)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
acc = accuracy_score(y_true, y_pred)
print(f"\nOverall Test Accuracy: {acc:.4f}\n")
print(report)

with open(os.path.join(RESULTS_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(f"Test Accuracy: {acc:.4f}\n\n")
    f.write(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, "%d" % cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
mis_idx = np.where(y_true != y_pred)[0]
idx = mis_idx[:10]
cols, rows = 5, 2
fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2.4))
axes = np.atleast_1d(axes).ravel()
for ax_i, i in enumerate(idx):
    axes[ax_i].imshow(x_test[i])
    axes[ax_i].set_title(f"T:{CLASS_NAMES[y_true[i]]}\nP:{CLASS_NAMES[y_pred[i]]}",
                         fontsize=8, color="red")
    axes[ax_i].axis("off")
for ax_i in range(len(idx), len(axes)):
    axes[ax_i].axis("off")
plt.suptitle("Misclassified Samples", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "misclassified.png"), dpi=120, bbox_inches="tight")
plt.show()

## 6) Save the Model

In [ ]:
model.save(os.path.join(MODELS_DIR, "cifar10_cnn_final.keras"))
print("Saved:", os.path.join(MODELS_DIR, "cifar10_cnn_final.keras"))

## 7) (Optional) Save Everything to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/CIFAR10_Project"
!cp -r models results "/content/drive/MyDrive/CIFAR10_Project/"
print("Saved to /content/drive/MyDrive/CIFAR10_Project/")